In [6]:
OUT = RAW / "codenet_cpp_metadata.parquet"

# Fix 3: Delete the partial output from the failed run
if OUT.exists():
    OUT.unlink()
    print("Removed previous partial output.\n")

writer = None
lang_status = Counter()
rows_written = 0
t0 = time.time()

try:
    for i, f in enumerate(files, 1):
        # Fix 2: Apply the pinned dtypes here
        df = pd.read_csv(f, usecols=USECOLS, dtype=DTYPES)
        lang_status.update(zip(df["language"], df["status"]))

        cpp = df[df["language"] == "C++"]
        if len(cpp):
            table = pa.Table.from_pandas(cpp, preserve_index=False)
            if writer is None:
                writer = pq.ParquetWriter(OUT, table.schema, compression="zstd")
            writer.write_table(table)
            rows_written += len(cpp)

        if i % 250 == 0:
            print(f"  {i:>4}/{len(files)}  C++ rows: {rows_written:>10,}  "
                  f"elapsed: {time.time()-t0:,.0f}s")
finally:
    if writer is not None:
        writer.close()

print(f"\nDone. {rows_written:,} C++ rows in {time.time()-t0:,.0f}s")
print(f"Output size: {OUT.stat().st_size/1e6:.1f} MB")

Removed previous partial output.

   250/4053  C++ rows:    194,563  elapsed: 3s
   500/4053  C++ rows:    292,995  elapsed: 6s
   750/4053  C++ rows:    376,751  elapsed: 7s
  1000/4053  C++ rows:    430,053  elapsed: 9s
  1250/4053  C++ rows:    470,709  elapsed: 10s
  1500/4053  C++ rows:    525,832  elapsed: 12s
  1750/4053  C++ rows:    563,126  elapsed: 13s
  2000/4053  C++ rows:    591,931  elapsed: 14s
  2250/4053  C++ rows:    626,120  elapsed: 16s
  2500/4053  C++ rows:  1,006,290  elapsed: 21s
  2750/4053  C++ rows:  2,619,224  elapsed: 37s
  3000/4053  C++ rows:  4,165,399  elapsed: 56s
  3250/4053  C++ rows:  5,378,884  elapsed: 70s
  3500/4053  C++ rows:  6,444,156  elapsed: 83s
  3750/4053  C++ rows:  7,262,189  elapsed: 94s
  4000/4053  C++ rows:  7,814,899  elapsed: 103s

Done. 8,008,527 C++ rows in 105s
Output size: 155.1 MB


In [7]:
df = pd.read_parquet(OUT)
print(f"Rows: {len(df):,}   (IBM published: 8,008,527)")
print(f"Memory: {df.memory_usage(deep=True).sum()/1e9:.2f} GB")
print(f"\nLanguages present: {df['language'].unique()}")
print(f"\nVerdicts:\n{df['status'].value_counts().to_string()}")

Rows: 8,008,527   (IBM published: 8,008,527)
Memory: 3.49 GB

Languages present: <StringArray>
['C++']
Length: 1, dtype: string

Verdicts:
status
Accepted                  4353049
Wrong Answer              2571284
Compile Error              376053
Runtime Error              339670
Time Limit Exceeded        326340
WA: Presentation Error      26449
Memory Limit Exceeded       14637
Output Limit Exceeded         778
Judge Not Available            94
Query Limit Exceeded           88
Internal error                 78
Judge System Error              7


In [8]:
ls = (pd.DataFrame([(l, s, c) for (l, s), c in lang_status.items()],
                   columns=["language", "status", "count"])
        .sort_values("count", ascending=False))
ls.to_csv(RAW / "language_status_counts.csv", index=False)

shutil.copy(META / "problem_list.csv", RAW / "problem_list.csv")

for p in sorted(RAW.iterdir()):
    if p.is_file():
        print(f"{p.name:<38} {p.stat().st_size/1e6:>8.1f} MB")

.gitkeep                                    0.0 MB
codenet_cpp_metadata.parquet              155.1 MB
language_status_counts.csv                  0.0 MB
problem_list.csv                            0.2 MB


In [3]:
from pathlib import Path
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from collections import Counter
import time, shutil

META = Path(r"D:\Datasets\CodeNet\extracted\Project_CodeNet\metadata")
RAW  = Path.cwd().parent / "data" / "raw"

DTYPES = {
    "submission_id":     "string",
    "problem_id":        "string",
    "user_id":           "string",
    "date":              "Int64",
    "language":          "string",
    "original_language": "string",
    "filename_ext":      "string",
    "status":            "string",
    "cpu_time":          "Int64",
    "memory":            "Int64",
    "code_size":         "Int64",
}
USECOLS = list(DTYPES)

files = sorted(META.glob("p[0-9]*.csv"))
print(f"Problem CSVs found: {len(files)}")
print(f"Output directory:   {RAW}  (exists: {RAW.exists()})")

Problem CSVs found: 4053
Output directory:   d:\Dev\Github\transformer-defect-prediction\data\raw  (exists: True)
